# G1 Academy Day 3 - Task 4: Build an IK Dashboard with Codex (Unsolved)

## Introduction
Every other task today had you call G1 SDK functions directly from a notebook. This task is different: instead of writing robot-control code by hand, you'll use **Codex** (an AI coding agent, run from a terminal) to build a small web dashboard on top of an *existing* script -- `/home/EF/ef_ws/g1/WBC/ik_pose_cli_v3.py`.

`ik_pose_cli_v3.py` is a terminal (curses) UI that controls the end effector's full 6D Cartesian pose (x, y, z, roll, pitch, yaw) via the same kind of incremental IK step you used in Task 3 (`ik_move_ee`), one key press at a time. Your goal is a **dashboard version** of it with a more ergonomic control layout:

- **x / y** of the end effector -> one on-screen **joystick** widget (drag/touch, both axes at once)
- **z** of the end effector -> two discrete **buttons** ("Z Up" / "Z Down")
- **roll / pitch / yaw** of the end effector -> three **sliders**, one per axis

You are not writing the IK or the safety clamps yourself -- they already exist in `ik_pose_cli_v3.py`. Your job (with Codex's help) is to read that file, reuse it correctly, and wire it to the new controls.

## What you're working with
Two files in `g1/WBC/` matter before you write a prompt:

- **`ik_pose_cli_v3.py`** -- the class `IKPoseCLI` holds all the control logic: `DOF_NAMES = ("x", "y", "z", "roll", "pitch", "yaw")`, an internal `dof_idx` selecting which DOF is "active", and `_adjust_dof(delta)`, which nudges the active DOF on the active arm(s) by `delta`, running IK and applying the same per-step Cartesian/joint-delta safety clamps you saw in Task 3. `tick()` must be called repeatedly (on a timer or background thread) to actually ramp toward the last-solved target and publish it. `_release_arms()` / `_unrelease_arms()` are the handoff helpers (same idea as `release_arms()`/`engage_arms()` from Task 1). A lock file prevents two processes from fighting over the arm at once.
- **`ik_pose_dash.py`** and **`ik_pose_dash_v2.py`** -- existing Dash web apps that *already* wrap `IKPoseCLI` in a browser UI, using generic +/- step buttons for all six DOF (`from ik_pose_cli_v3 import IKPoseCLI, ...`). Read `ik_pose_dash_v2.py` first: it's your template for "how do I turn a UI event into a call into `IKPoseCLI`" -- you're building the same kind of app, just with a joystick + 2 buttons + 3 sliders instead of 12 step buttons.

These are internal (`_`-prefixed) methods, not a documented public API -- reuse them the way the existing dashboards already do, rather than re-deriving the IK or the ramping logic yourself.

## Safety constraints your dashboard must keep
These limits already live inside `ik_pose_cli_v3.py` -- your dashboard should respect them, not work around them:

- **Per-step clamps.** Every call into the control loop clamps the Cartesian step and the resulting joint delta, the same way `ik_move_ee` did in Task 3. Don't add a "big jump" shortcut that bypasses `_adjust_dof`/`tick()`.
- **Single owner.** Only one process may hold the pose lock / own `rt/arm_sdk` at a time. Your dashboard must go through the CLI's existing release/engage path rather than opening a second, competing publisher.
- **`kd` ceiling.** `sdk_wrapper.py` hard-clamps every commanded joint's `kd` to `MAX_KD = 10.0` no matter what's requested. This task doesn't ask for a gain/stiffness control, but if you (or Codex) ever add one later, clamp it client-side too, so the UI never implies a value the robot won't actually honor.
- **A reachable STOP.** Add a visible, always-available "STOP / release arms" control. Never ship a version where the only way to stop a runaway joystick input is to close the browser tab.

## Using Codex
Run `codex` from a terminal in `g1/WBC/` (or point it at the repo root and mention the path). Give it a clear, scoped prompt, **review every diff it proposes before running it**, and test against the robot in `damp` / on a stand before trying anything free-standing.

Example prompt:

```text
I'm working in g1/WBC/. Read ik_pose_cli_v3.py (class IKPoseCLI) and
ik_pose_dash_v2.py (an existing Dash web UI that wraps IKPoseCLI with
+/- step buttons for all 6 DOF) to understand the pattern.

Create a new file, ik_pose_joystick_dash.py, that wraps IKPoseCLI in a
Dash app with this control layout instead:

- A 2D joystick widget (drag or touch) that continuously streams
  (dx, dy) while held, mapped to the x and y DOF (index 0 and 1 in
  DOF_NAMES) via the same _adjust_dof()-style step used by the
  existing dashboards -- do not bypass the per-step Cartesian clamp
  already inside ik_pose_cli_v3.py.
- Two buttons, "Z Up" and "Z Down", that each nudge the z DOF (index 2)
  by one step per click (and optionally repeat while held).
- Three sliders, one each for roll, pitch, and yaw (DOF indices 3, 4,
  5), where each slider change is applied as an incremental adjustment
  from its previous value, not sent as an absolute target -- follow
  how ik_pose_dash_v2.py already turns a UI event into a delta call.
- Keep the hand selector (left/right/both), the release/re-engage arm
  buttons, and a visible STOP button that calls the same
  release-arms path as the 'r' key in the CLI.
- Reuse IKPoseCLI's existing tick()/background-thread pattern from
  ik_pose_dash_v2.py rather than writing a new control loop.

Don't change ik_pose_cli_v3.py. Ask me before running anything against
the real robot.
```

Adjust the prompt as you go -- e.g. tell Codex what actually happened when you ran the app, and let it iterate.

## Deliverable / how to check your work
Run the generated app and confirm, in this order:

1. The joystick moves x/y smoothly as you drag it (small continuous steps, not one giant jump per drag).
2. The two z buttons each move the end effector by one small, fixed increment per click.
3. Each slider moves the corresponding orientation axis (roll/pitch/yaw) as you drag it.
4. The STOP button reliably releases the arm (equivalent to `release_arms()`/the CLI's `r` key) -- test this *before* you rely on it for anything else.
5. Nothing in the new file changes or bypasses the clamps in `ik_pose_cli_v3.py`.

Review the diff Codex produced end-to-end before your first real run, the same as you would for a teammate's pull request.

In [ ]:
# After Codex creates g1/WBC/ik_pose_joystick_dash.py, smoke-test it
# (from a terminal, or by shelling out from here) before trying it against
# the real robot, e.g.:
#   python3 ik_pose_joystick_dash.py --help

# TODO: Implement this section using the preceding task description and day slides.
# Keep robot commands conservative; test observation/state code before actuation.
raise NotImplementedError("Complete this task section")


### Safety
Review every diff Codex proposes before running it -- never let generated code run unattended against the physical robot. Keep FSM `damp` reachable, test on a stand first, and confirm the STOP control in your dashboard actually reaches the release-arms path before you rely on it. Code in this notebook is never invoked automatically -- you decide when each cell runs.